In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from wh_research.data.hyperliquid.labels import add_forward_returns
from wh_research.data.hyperliquid.features import add_past_returns

# Computing labels

### I will choose the following labels as candidates

-  1-day forward log-return
-  3-day forward log-return
-  7-day forward log-return
-  1-day forward log-return direction
-  3-day forward log-return direction
-  7-day forward log-return direction

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent
data_path = PROJECT_ROOT/"data/hyperliquid/case_study_data/universe_1d_min_mdv_2m_exit_1m_280323_171026.parquet"
df = pd.read_parquet(data_path)

### Computing the labels

`add_forward_returns` adds, for each horizon n, three columns:
- `fwd_log_return_{n}`: log(close n days later / this close), from this bar's `close_time`
- `fwd_direction_{n}`: 1 if that return is above 0, 0 if not, and missing where the return is missing
- `fwd_end_time_{n}`: when the label becomes known (used to purge the splits)

We compute them on every row first and only then keep the rows where `in_universe` is True, so the labels never skip over days when a symbol was outside the universe. The `fwd_` columns look into the future by design: they are never used as features.

In [3]:
# Add the forward log-return, direction and end-time columns for 1, 3 and 7 days.
# Done on the complete series of every symbol (all rows), before filtering to the universe.
df = add_forward_returns(df, "1d", [1, 3, 7])

#### Check the labels on the days in the universe

The last n days of every symbol have no label yet (the future has not happened), so they are missing. We do not drop them here.

In [4]:
labelled = df[df["in_universe"]]
horizons = [1, 3, 7]

print("Rows in the universe:", len(labelled))
print()
print("Missing labels:")
print(labelled[[f"fwd_log_return_{n}" for n in horizons]].isna().sum())
print()
print("Share of up days (missing labels skipped):")
print(labelled[[f"fwd_direction_{n}" for n in horizons]].mean())

Rows in the universe: 59645

Missing labels:
fwd_log_return_1     71
fwd_log_return_3    212
fwd_log_return_7    492
dtype: int64

Share of up days (missing labels skipped):
fwd_direction_1    0.469937
fwd_direction_3    0.459021
fwd_direction_7    0.437036
dtype: float64


# Computing features

### I will choose the following features as candidates

-  7-day log-return
-  14-day log-return
-  30-day log-return

### Computing the features

`add_past_returns` adds, for each horizon n, `past_log_return_{n}`: log(close today / close n days ago).

It only uses the close at this bar's `close_time` and the close n days earlier, so it is known at the same moment as the `in_universe` flag and is used on the same row with no shift. Like the labels, we compute them on every row first and only then keep the universe rows.

In [5]:
# Add the 7, 14 and 30-day past log-returns. Done on the complete series of every symbol (all rows), before filtering to the universe.
df = add_past_returns(df, "1d", [7, 14, 30])

#### Check the features on the days in the universe

A symbol only enters the universe on its 31st bar, so every universe row has the 30 days of history it needs and nothing should be missing.

In [6]:
members = df[df["in_universe"]]
features = ["past_log_return_7", "past_log_return_14", "past_log_return_30"]

print("Rows in the universe:", len(members))
print()
print("Missing features:")
print(members[features].isna().sum())
print()
print(members[features].describe())

Rows in the universe: 59645

Missing features:
past_log_return_7     0
past_log_return_14    0
past_log_return_30    0
dtype: int64

       past_log_return_7  past_log_return_14  past_log_return_30
count       59645.000000        59645.000000        59645.000000
mean           -0.015435           -0.027515           -0.040150
std             0.156321            0.222031            0.354923
min            -2.359258           -2.541866           -2.874894
25%            -0.100300           -0.150200           -0.242652
50%            -0.015655           -0.030068           -0.045218
75%             0.062571            0.084174            0.130989
max             1.531704            2.021841            2.663073


# Splitting off the holdout

We use the same date boundary as in `2_determining_cv_folds` (cut by date: the last 30% of the days that have symbols in the universe are the holdout).

In [7]:
# every day with at least one symbol in the universe, oldest first
days = df.loc[df["in_universe"], "open_time"].drop_duplicates().sort_values().reset_index(drop=True)

# pre-holdout = first 70% of the days, holdout = last 30% of the days
holdout_start = days.iloc[int(len(days) * 0.70)]

print(f"Holdout starts : {holdout_start}")

Holdout starts : 2025-09-22 00:00:00+00:00


### Purged split

The longest label is 7 days ahead, so `n = 7`. A pre-holdout row is kept only if its label is fully known (`fwd_end_time_7`) before the holdout starts, otherwise its label would look into the holdout.

- `pre_holdout`: used for feature evaluation and all research. The CV folds are built from it later, after the features and the label are chosen
- `holdout`: everything from the holdout start. It is only for the final unbiased evaluation, so do not open it while developing models

Rows with a missing label (the last days of every symbol) are kept. The modelling step decides what to do with them.

In [8]:
# the longest label horizon we use, in days
n = 7
end = f"fwd_end_time_{n}"

# pre-holdout: labels fully known before the holdout starts
pre_holdout = members[members[end] < holdout_start]
# holdout: everything from the holdout start
holdout = members[members["open_time"] >= holdout_start]

# every pre-holdout label must end before the holdout starts
assert pre_holdout[end].max() < holdout_start

sets = {"pre_holdout": pre_holdout, "holdout": holdout}

# a short summary of each set
summary = pd.DataFrame({
    name: {
        "rows": len(part),
        "symbols": part["symbol"].nunique(),
        "first open_time": part["open_time"].min(),
        "last open_time": part["open_time"].max(),
    }
    for name, part in sets.items()
}).T
summary

,rows,symbols,first open_time,last open_time
pre_holdout,37725,143,2023-03-28 00:00:00+00:00,2025-09-14 00:00:00+00:00
holdout,21454,110,2025-09-22 00:00:00+00:00,2026-10-17 00:00:00+00:00


### Saving the sets

Each set is saved next to the universe file in `data/hyperliquid/case_study_data`, with all columns (candle columns, `in_universe`, the `fwd_*` labels and the `past_log_return_*` features).

In [9]:
save_dir = PROJECT_ROOT/"data/hyperliquid/case_study_data"

for name, part in sets.items():
    part.to_parquet(save_dir/f"{name}.parquet", index=False)
    print(f"saved {name}.parquet ({len(part)} rows)")

saved pre_holdout.parquet (37725 rows)
saved holdout.parquet (21454 rows)
